### Infernce time 

In [13]:
# ===========================
# 1. IMPORTS
# ===========================
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input


# ===========================
# 2. LOAD DATA
# ===========================
X_images = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_images.npy")
X_rna = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_rna.npy")
X_clinical = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_clinical.npy")
y = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\y.npy")

print("Loaded:", X_images.shape, X_rna.shape, X_clinical.shape)


# ===========================
# 3. RNA REDUCTION
# ===========================
pca = PCA(n_components=20)
X_rna = pca.fit_transform(X_rna)


# ===========================
# 4. IMAGE FEATURE EXTRACTION (PRETRAINED)
# ===========================
n_patients, n_slices = X_images.shape[0], X_images.shape[1]

X_img = X_images.reshape(-1, 224, 224, 1)
X_img = np.repeat(X_img, 3, axis=-1)
X_img = preprocess_input(X_img)

base_model = MobileNetV3Small(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False

# 🔥 TIMING IMAGE FEATURE EXTRACTION
start_img_time = time.time()

features = base_model.predict(X_img, verbose=1)

end_img_time = time.time()
print(f"\nImage Feature Extraction Time: {end_img_time - start_img_time:.4f} sec")

features = np.mean(features, axis=(1,2))
features = features.reshape(n_patients, n_slices, -1)

X_img_features = np.mean(features, axis=1)

pca_img = PCA(n_components=20)
X_img_features = pca_img.fit_transform(X_img_features)

print("Image features:", X_img_features.shape)


# ===========================
# 5. CLINICAL FEATURES
# ===========================
X_clinical_final = X_clinical


# ===========================
# 6. FUSION
# ===========================
X_fused = np.concatenate([
    X_img_features,
    X_rna,
    X_clinical_final
], axis=1)

scaler = StandardScaler()
X_fused = scaler.fit_transform(X_fused)

print("Fused shape:", X_fused.shape)


# ===========================
# 7. MODEL DEFINITION
# ===========================
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# ===========================
# 8. CROSS VALIDATION
# ===========================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
f1_scores = []
inference_times = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_fused, y)):
    print(f"\n===== FOLD {fold+1} =====")

    X_train, X_test = X_fused[train_idx], X_fused[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    model = SimpleNN(X_fused.shape[1])

    pos_weight = torch.tensor([(len(y_train) - sum(y_train)) / sum(y_train)])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # TRAIN
    for epoch in range(80):
        model.train()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # ===========================
    # 🔥 FIXED INFERENCE TIMING
    # ===========================
    model.eval()

    runs = 50   # stabilize timing
    total_time = 0

    with torch.no_grad():
        for _ in range(runs):
            start = time.perf_counter()
            logits = model(X_test_t)
            end = time.perf_counter()
            total_time += (end - start)

        preds = torch.sigmoid(logits).numpy()

    inference_time = total_time / (runs * len(X_test_t))
    inference_times.append(inference_time)

    print(f"Inference Time per sample: {inference_time:.6f} sec")

    # ===========================
    # threshold tuning
    # ===========================
    best_f1 = 0
    best_thresh = 0.5

    for t in np.arange(0.1, 0.9, 0.05):
        temp_preds = (preds > t).astype(int)
        f1 = f1_score(y_test, temp_preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    final_preds = (preds > best_thresh).astype(int)

    acc = accuracy_score(y_test, final_preds)
    f1 = f1_score(y_test, final_preds)

    print(f"Best Threshold: {best_thresh:.2f}")
    print(f"Accuracy: {acc:.3f}, F1: {f1:.3f}")

    acc_scores.append(acc)
    f1_scores.append(f1)
# ===========================
# 9. FINAL RESULTS
# ===========================
print("\n===== FINAL RESULTS =====")
print("Mean Accuracy:", np.mean(acc_scores))
print("Mean F1 Score:", np.mean(f1_scores))


# ===========================
# 🔥 FINAL INFERENCE SUMMARY
# ===========================
print("\n===== INFERENCE TIME SUMMARY =====")
print("Mean Inference Time:", np.mean(inference_times))
print("Min Inference Time:", np.min(inference_times))
print("Max Inference Time:", np.max(inference_times))

Loaded: (42, 50, 224, 224, 1) (42, 20) (42, 76)
66/66 ━━━━━━━━━━━━━━━━━━━━ 11s 140ms/step

Image Feature Extraction Time: 18.4862 sec
Image features: (42, 20)
Fused shape: (42, 116)

===== FOLD 1 =====
Inference Time per sample: 0.000006 sec
Best Threshold: 0.35
Accuracy: 0.667, F1: 0.571

===== FOLD 2 =====
Inference Time per sample: 0.000006 sec
Best Threshold: 0.60
Accuracy: 0.889, F1: 0.857

===== FOLD 3 =====
Inference Time per sample: 0.000007 sec
Best Threshold: 0.10
Accuracy: 0.375, F1: 0.545

===== FOLD 4 =====
Inference Time per sample: 0.000008 sec
Best Threshold: 0.65
Accuracy: 0.875, F1: 0.667

===== FOLD 5 =====
Inference Time per sample: 0.000005 sec
Best Threshold: 0.55
Accuracy: 0.750, F1: 0.667

===== FINAL RESULTS =====
Mean Accuracy: 0.711111111111111
Mean F1 Score: 0.6614718614718613

===== INFERENCE TIME SUMMARY =====
Mean Inference Time: 6.130983333175917e-06
Min Inference Time: 4.953000001819419e-06
Max Inference Time: 7.609999997839623e-06
